In [ ]:
# =============================================================================
# Neural-SINDy  —  Sharma et al. 2022 CONSISTENT VERSION (FULL OUTPUT FORMAT)
# =============================================================================

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.fft import rfft, rfftfreq
from scipy.signal import butter, filtfilt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
np.random.seed(0)

# ─────────────────────────────────────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────────────────────────────────────
def load_data(lift_path="drag_lift_body_001.dat",
              struct_path="viv_body1_probe.dat"):

    lift = pd.read_csv(lift_path, sep=r"\s+", skiprows=1,
                       names=["t","cxp","cxs","cx","cyp","cys","cy"])
    stru = pd.read_csv(struct_path, sep=r"\s+", skiprows=2,
                       names=["nt","t","DX","DY","VY","AY"])

    t  = stru["t"].values
    y  = stru["DY"].values
    CT = np.interp(t, lift["t"].values, lift["cy"].values)

    t -= t[0]

    s = int(0.3 * len(t))
    t, y, CT = t[s:], y[s:], CT[s:]

    b, a = butter(4, 0.12)
    y  = filtfilt(b, a, y)
    CT = filtfilt(b, a, CT)

    return t, y, CT


def dominant_freq(t, sig):
    dt = np.mean(np.diff(t))
    Y  = rfft(sig - sig.mean()); Y[0] = 0
    f  = rfftfreq(len(sig), dt)[np.argmax(np.abs(Y))]
    return float(f), float(2 * np.pi * f), float(1 / f)


# ─────────────────────────────────────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────────────────────────────────────
class SineLayer(nn.Module):
    def __init__(self, in_f, out_f, w0=20.):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f)
        self.w0 = w0

    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


class SIREN(nn.Module):
    def __init__(self, width=64, depth=3):
        super().__init__()
        layers = [SineLayer(1, width)]
        for _ in range(depth - 1):
            layers.append(SineLayer(width, width))
        self.net = nn.Sequential(*layers)
        self.out = nn.Linear(width, 1)

    def forward(self, t):
        return self.out(self.net(t))


# ─────────────────────────────────────────────────────────────────────────────
# PHYSICS
# ─────────────────────────────────────────────────────────────────────────────
class PhysParams(nn.Module):

    M_RATIO = 5.0
    XI = 0.0

    def __init__(self, f_n):
        super().__init__()

        self.m = self.M_RATIO
        self.k = self.m * (2 * np.pi * f_n) ** 2

        # Learnable (correct sign physics enforced in model, not here)
        self.a1 = nn.Parameter(torch.tensor(0.14))
        self.a3 = nn.Parameter(torch.tensor(-5.0))


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():

    print("\n================ D-SECTION VIV & GALLOPING IDENTIFICATION ================")
    print("       ")

    t_np, y_np, CT_np = load_data()

    f_n, omega_n, T_n = dominant_freq(t_np, y_np)

    Tmax  = t_np[-1]
    Yscl  = np.std(y_np)
    CTscl = np.std(CT_np)

    tn     = torch.tensor(t_np / Tmax, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    y_dat  = torch.tensor(y_np / Yscl, dtype=torch.float32).unsqueeze(1).to(DEVICE)
    CT_dat = torch.tensor(CT_np / CTscl, dtype=torch.float32).unsqueeze(1).to(DEVICE)

    idx = np.arange(0, len(tn), 4)
    tn     = tn[idx].clone().detach().requires_grad_(True)
    y_dat  = y_dat[idx]
    CT_dat = CT_dat[idx]

    net_y = SIREN().to(DEVICE)
    P     = PhysParams(f_n).to(DEVICE)

    params = list(net_y.parameters()) + [P.a1, P.a3]
    opt = torch.optim.Adam(params, lr=3e-4)

    def ddt(u, t):
        return torch.autograd.grad(u.sum(), t, create_graph=True)[0]

    print("[Training…]")
    for ep in range(1500):

        opt.zero_grad()

        yh  = net_y(tn)
        yh1 = ddt(yh, tn)
        yh2 = ddt(yh1, tn)

        # physical variables
        y_phys   = yh * Yscl
        ydot     = yh1 * (Yscl / Tmax)
        yddot    = yh2 * (Yscl / Tmax**2)

        # ── Loss A
        LA = torch.mean((yh - y_dat)**2)

        # ── FIXED galloping physics
        U_rel = -ydot
        CT_mod = (P.a1 * U_rel + P.a3 * U_rel**3) / CTscl

        LB = torch.mean((CT_mod - CT_dat)**2)

        # ── Structural equation
        rS = (P.m * yddot + P.k * y_phys - CT_mod * CTscl)
        LC = torch.mean(rS**2)

        loss = LA + LB + 0.02 * LC
        loss.backward()

        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()

        tn = tn.detach().requires_grad_(True)

        if ep % 200 == 0:
            print(f"ep {ep:4d} | loss {loss.item():.4e}")

    # ─────────────────────────────────────────────────────────────────────────
    # RESULTS
    # ─────────────────────────────────────────────────────────────────────────
    a1 = P.a1.item()
    a3 = P.a3.item()
    m  = P.m
    k  = P.k

    print("\n================ RESULTS & VERIFICATION =================\n")

    print("Kinematics:")
    print(f"  Frequency f_n      : {f_n:.6f} Hz")
    print(f"  Angular freq ω_n   : {omega_n:.6f} rad/s")
    print(f"  Period T           : {T_n:.6f} s\n")

    print("Structural Properties")
    print(f"  Mass ratio (m)     : {m:.5f}")
    print(f"  Damping ratio (ξ)  : {PhysParams.XI:.5f}   ")
    print(f"  Stiffness (k)      : {k:.5f}   [= m·(2πf_n)²]\n")

    print("Galloping Coefficients :")
    print(f"  a1 (Linear) = β    : {a1:.5f} ")
    print(f"  a3 (Cubic)         : {a3:.5f}\n")

    print("================ GOVERNING EQUATIONS =================\n")
    print("  ")
    print(f"  {m:.3f} y'' + {k:.3f} y  =  C_T\n")

    print("  [Quasi-steady galloping force ")
    print(f"  C_T  =  {a1:.3f} y'  +  {a3:.3f} (y')^3\n")

    print("  Den Hartog criterion  β = a1 > 0  → galloping")
    print(f"  β = {a1:.4f}  →  {'GALLOPING tendency confirmed' if a1 > 0 else 'No galloping tendency'}")


# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    main()


================ D-SECTION VIV & GALLOPING IDENTIFICATION ================
       
[Training…]
ep    0 | loss 2.1429e+00
ep  200 | loss 1.1048e+00
ep  400 | loss 1.0963e+00
ep  600 | loss 1.0870e+00
ep  800 | loss 1.0900e+00
ep 1000 | loss 1.0874e+00
ep 1200 | loss 1.0878e+00
ep 1400 | loss 1.0877e+00

================ RESULTS & VERIFICATION =================

Kinematics:
  Frequency f_n      : 0.191429 Hz
  Angular freq ω_n   : 1.202781 rad/s
  Period T           : 5.223881 s

Structural Properties
  Mass ratio (m)     : 5.00000
  Damping ratio (ξ)  : 0.00000   
  Stiffness (k)      : 7.23341   [= m·(2πf_n)²]

Galloping Coefficients :
  a1 (Linear) = β    : 0.00843 
  a3 (Cubic)         : -4.70450

================ GOVERNING EQUATIONS =================

  
  5.000 y'' + 7.233 y  =  C_T

  [Quasi-steady galloping force 
  C_T  =  0.008 y'  +  -4.705 (y')^3

  Den Hartog criterion  β = a1 > 0  → galloping
  β = 0.0084  →  GALLOPING tendency confirmed
